In [73]:
import pandas as pd

# Reading the both csv files
beije = pd.read_csv("beije.csv")
form = pd.read_csv("studying_behavior_form_responses.csv")
form.columns = form.columns.str.strip()
beije.columns=beije.columns.str.strip()


In [74]:
beije_df=pd.DataFrame(beije)
beije_df.head()

,cycle_start,cycle_end
0,28.06.2025,05.07.2025
1,25.07.2025,NaN
2,25.07.2025,31.07.2025
3,25.08.2025,31.08.2025
4,22.09.2025,29.09.2025


In [75]:
form_df=pd.DataFrame(form)
form_df.head()

,date,painkiller_usage,study_hours_daily,exam_period,coffee_consumption
0,01.06.2025,No,05:00:00,No,yes
1,02.06.2025,yes,07:00:00,yes,yes
2,03.06.2025,No,04:00:00,yes,yes
3,04.06.2025,No,08:00:00,yes,yes
4,05.06.2025,No,04:00:00,yes,yes


In [76]:
# Beije tarihleri 
beije['cycle_start'] = pd.to_datetime(beije['cycle_start'], dayfirst=True, errors='coerce')
beije['cycle_end'] = pd.to_datetime(beije['cycle_end'], dayfirst=True, errors='coerce')

#goggle form tarihlerini dayfirst'e göre formatladık
form['date']=pd.to_datetime(form['date'], dayfirst= True, errors='coerce')

#tarihleri kronolojik olarak sıraya soktuk
form=form.sort_values('date')
beije=beije.sort_values('cycle_start')



In [77]:
#beije dosyasını günlük seviyeye genişlettik ki form dosyasıyla tarih eşleştirmesi yapabilelim
#ve de is_period adında bir sütun ekledik
expanded = []

for _, row in beije.iterrows():
    if pd.isna(row['cycle_start']) or pd.isna(row['cycle_end']):
        continue
        
    dates = pd.date_range(start=row['cycle_start'], end=row['cycle_end'])

    for d in dates:
        expanded.append({
            'date': d,
            'is_period': 1 
        })

beije_expanded = pd.DataFrame(expanded)

#beije_expanded ile form'u birleştirdik
df = form.merge(beije_expanded, on='date', how='left')

#period tarihi olmayıp boş kalan is_period satırlarına 0 değerini atadık
df['is_period'] = df['is_period'].fillna(0).astype(int)
df.head(60)


,date,painkiller_usage,study_hours_daily,exam_period,coffee_consumption,is_period
0,2025-06-01,No,05:00:00,No,yes,0
1,2025-06-02,yes,07:00:00,yes,yes,0
2,2025-06-03,No,04:00:00,yes,yes,0
3,2025-06-04,No,08:00:00,yes,yes,0
4,2025-06-05,No,04:00:00,yes,yes,0
5,2025-06-06,No,00:00:00,No,yes,0
6,2025-06-07,No,00:00:00,No,yes,0
7,2025-06-08,yes,00:00:00,No,no,0
8,2025-06-09,No,00:00:00,No,yes,0
9,2025-06-10,yes,00:00:00,No,yes,0


In [78]:
#tekrar eden satırlarda en son giirlmiş olan satırı doğru kabul edip diğer girişleri sildik
df = df.drop_duplicates(subset='date', keep='last')
df = df.reset_index(drop=True)

In [79]:
#boş veri girişlerini kontrol ettik
df.isna().sum()

date                  0
painkiller_usage      0
study_hours_daily     0
exam_period           0
coffee_consumption    0
is_period             0
dtype: int64

In [80]:
#boolean değerli olması gereken kolonlardaki girdileri 0 ve 1'lere çevirdik
bool_cols = ['painkiller_usage', 'exam_period', 'coffee_consumption']

for col in bool_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

mapping = {
    "yes": 1,
    "no": 0,
    "1": 1,
    "0": 0,
    "true": 1,
    "false": 0
}

for col in bool_cols:
    df[col] = df[col].map(mapping)

df.head(20)

,date,painkiller_usage,study_hours_daily,exam_period,coffee_consumption,is_period
0,2025-06-01,0,05:00:00,0,1,0
1,2025-06-02,1,07:00:00,1,1,0
2,2025-06-03,0,04:00:00,1,1,0
3,2025-06-04,0,08:00:00,1,1,0
4,2025-06-05,0,04:00:00,1,1,0
5,2025-06-06,0,00:00:00,0,1,0
6,2025-06-07,0,00:00:00,0,1,0
7,2025-06-08,1,00:00:00,0,0,0
8,2025-06-09,0,00:00:00,0,1,0
9,2025-06-10,1,00:00:00,0,1,0


In [ ]:
#günlük çalışma saati verilerini float veri tipine dönüştürüp saat cinsinden yazdık (ör: 3 saat 45 dakika = 3.75)
df['study_hours_daily'] = pd.to_timedelta(df['study_hours_daily'], errors='coerce')
df['study_hours_daily'] = df['study_hours_daily'].dt.total_seconds() / 3600
df['study_hours_daily'] = df['study_hours_daily'].round(2)

#deleting the outliers
df = df[(df['study_hours_daily'] >= 0) & (df['study_hours_daily'] <= 16)]

df[(df['date'] >= "2025-09-10") & (df['date'] <= "2025-11-10")]

,date,painkiller_usage,study_hours_daily,exam_period,coffee_consumption,is_period
101,2025-09-10,0,0.00,0,1,0
102,2025-09-11,0,0.00,0,0,0
103,2025-09-12,0,0.00,0,1,0
104,2025-09-13,0,0.00,0,1,0
106,2025-09-15,0,0.00,0,0,0
107,2025-09-16,0,0.00,0,1,0
108,2025-09-17,0,1.00,0,0,0
109,2025-09-18,0,0.00,0,0,0
110,2025-09-19,1,2.50,0,1,0
111,2025-09-20,0,0.00,0,0,0


In [106]:
#temizlenmiş vetri setini csv olarak kaydediyoruz.
df.to_csv("/Users/berilnurserbest/Desktop/DSA210_Project/data_cleaned.csv", index=False, encoding="utf-8")
